# 英国电力需求时序预测 —— 基于 Transformer

本项目使用英国国家电网的历史电力需求数据，通过 **Transformer** 模型进行时序预测。

主要步骤：
* **[EDA 与特征工程](#eda_fe)** — 数据清洗、特征创建、趋势与季节性分析
* **[Transformer 模型](#transformer)** — 使用 PyTorch 实现 Transformer 进行电力需求预测
* **[结果分析](#results)** — 模型评估与可视化

# 目录
* [EDA 与特征工程](#eda_fe)
* [Transformer 模型](#transformer)
* [结果分析](#results)

In [24]:
import datetime
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# 检测设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

使用设备: cuda


## 超参数配置

所有可调参数集中在此处，方便统一修改和实验对比。

In [ ]:
# ============================================================
# 超参数配置 —— 所有可调参数集中在此
# ============================================================

# --- 随机种子 ---
SEED = 221

# --- 数据划分 ---
THRESHOLD_DATE_1 = "01-01-2023"  # 训练集/验证集分界
THRESHOLD_DATE_2 = "01-01-2024"  # 验证集/测试集分界

# --- 序列窗口 ---
SEQ_LEN = 48  # 滑动窗口长度（48 = 一天的半小时采样数）

# --- DataLoader ---
BATCH_SIZE = 144

# --- Transformer 模型结构 ---
D_MODEL = 64           # 模型内部维度
NHEAD = 4              # 注意力头数
NUM_LAYERS = 2         # Transformer Encoder 层数
DIM_FEEDFORWARD = 128  # 前馈网络隐藏层维度
DROPOUT = 0.5          # Dropout 比率

# --- 训练 ---
NUM_EPOCHS = 100
LEARNING_RATE = 5e-4
PATIENCE = 12         # 早停耐心值
LR_FACTOR = 0.5        # 学习率衰减倍率
LR_PATIENCE = 5        # 学习率衰减耐心值

print("超参数配置完成")

<a id="eda_fe"></a>
## 图表默认参数

设置全局字体和字号，避免每次画图都重复设置。

In [26]:
# 设置默认字体和字号
small_size = 12
medium_size = 14
large_size = 16

matplotlib.rc("font", size=small_size)
matplotlib.rc("axes", titlesize=small_size)
matplotlib.rc("axes", labelsize=medium_size)
matplotlib.rc("xtick", labelsize=small_size)
matplotlib.rc("ytick", labelsize=small_size)
matplotlib.rc("legend", fontsize=small_size)
matplotlib.rc("axes", titlesize=large_size)

# 加载数据

使用英国国家电网 2019-2024 年的历史电力需求数据。


In [ ]:
df = pd.read_csv("data/historic_demand_2009_2024_noNaN.csv")

# 列名转小写
df.columns = df.columns.str.lower()

df.sample(n=7)

In [ ]:
df.describe()

In [ ]:
df.shape

<a id='eda'></a>
# EDA 与特征工程

## 数据理解

目标变量为 **TSD**（输电系统需求，单位 MW）。`settlement_period` 为每日的半小时采样周期（每天48个点）。

In [ ]:
# 按日期排序
df.sort_values(by=["settlement_date", "settlement_period"], inplace=True, ignore_index=True)

## 数据准备

- 删除含 NaN 的列（nsl_flow, eleclink_flow）
- 删除 settlement_period > 48 的异常行
- 删除 tsd 为 0 的异常天

In [ ]:
# 删除含大量缺失值的列（如果存在）
for col in ["nsl_flow", "eleclink_flow"]:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# 删除异常的 settlement_period
df.drop(index=df[df["settlement_period"] > 48].index, inplace=True)
df.reset_index(drop=True, inplace=True)

# 删除 tsd 为 0 的整天数据
null_days = df.loc[df["tsd"] == 0.0, "settlement_date"].unique().tolist()
null_days_index = []
for day in null_days:
    null_days_index.append(df[df["settlement_date"] == day].index.tolist())
null_days_index = [item for sublist in null_days_index for item in sublist]
if null_days_index:
    df.drop(index=null_days_index, inplace=True)
    df.reset_index(drop=True, inplace=True)

print(f"清洗后数据形状: {df.shape}")

## 设置日期时间索引

In [ ]:
# 将 settlement_date 设为日期时间索引
df["settlement_date"] = pd.to_datetime(df["settlement_date"])
df.set_index("settlement_date", inplace=True)
df.sort_index(inplace=True)

# 只保留 2019-2024 年的数据
df = df["2019-01-01":"2024-12-31"]

df.head()

## 特征创建

从日期索引中提取时间特征，并添加年度滞后特征。

In [ ]:
def create_features(df):
    """从时间索引中提取时间特征"""
    df = df.copy()
    df["day_of_month"] = df.index.day
    df["day_of_week"] = df.index.day_of_week
    df["day_of_year"] = df.index.day_of_year
    df["quarter"] = df.index.quarter
    df["month"] = df.index.month
    df["year"] = df.index.year
    df["week_of_year"] = df.index.isocalendar().week.astype("int64")
    return df

def add_lags(df):
    """添加过去三年的滞后特征"""
    target_map = df["tsd"].to_dict()
    df["lag1"] = (df.index - pd.Timedelta("364 days")).map(target_map)
    df["lag2"] = (df.index - pd.Timedelta("728 days")).map(target_map)
    df["lag3"] = (df.index - pd.Timedelta("1092 days")).map(target_map)
    return df

df = create_features(df)
print(f"添加 lag 特征并去除 NaN 后: {df.shape}")
df.sample(n=5)

## 特征分析

In [ ]:
# 每日不同时段的用电分布
fig, ax = plt.subplots(figsize=(15, 5))
ax.scatter(df["settlement_period"], df["tsd"], alpha=0.05, s=1)
ax.set_xlabel("Settlement Period (48 half-hours per day)")
ax.set_ylabel("Electricity Demand (MW)")
ax.set_title("Electricity Demand Distribution by Settlement Period");

In [ ]:
# 按年份的用电分布
fig, ax = plt.subplots(figsize=(15, 5))
yearly_mean = df.groupby("year")["tsd"].mean()
ax.bar(yearly_mean.index, yearly_mean.values)
ax.set_xlabel("Year")
ax.set_ylabel("Average Electricity Demand (MW)")
ax.set_title("Annual Average Electricity Demand");

In [ ]:
# 时间序列总览
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(df.index, df["tsd"], ".", markersize=0.5, alpha=0.3)
ax.set_xlabel("Date")
ax.set_ylabel("Electricity Demand (MW)")
ax.set_title("UK Electricity Demand Time Series Overview");

# 时序预测模型

## 评估指标

In [ ]:
def mean_absolute_percentage_error(y_true, y_pred):
    """计算平均绝对百分比误差 (MAPE)"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return mape

## 数据划分

将数据分为训练集、测试集和保留集（hold-out）。

In [ ]:
# 设置随机种子
torch.manual_seed(SEED)
np.random.seed(SEED)

train_data = df.loc[df.index < THRESHOLD_DATE_1].copy()
val_data = df.loc[(df.index >= THRESHOLD_DATE_1) & (df.index < THRESHOLD_DATE_2)].copy()
test_data = df.loc[df.index >= THRESHOLD_DATE_2].copy()

# ---- 构建 lag 特征：只用训练集的 tsd 作为来源，避免数据泄露 ----
target_map_train = train_data["tsd"].to_dict()
for subset in [train_data, val_data, test_data]:
    subset["lag1"] = (subset.index - pd.Timedelta("364 days")).map(target_map_train)
    subset["lag2"] = (subset.index - pd.Timedelta("728 days")).map(target_map_train)
    subset["lag3"] = (subset.index - pd.Timedelta("1092 days")).map(target_map_train)

# 对缺失的 lag 用前向填充，再用训练集均值填充剩余 NaN
train_lag_mean = train_data[["lag1", "lag2", "lag3"]].mean()
for subset in [train_data, val_data, test_data]:
    subset[["lag1", "lag2", "lag3"]] = subset[["lag1", "lag2", "lag3"]].fillna(train_lag_mean)

# 可视化数据划分
fig, ax = plt.subplots(figsize=(15, 5))
train_data["tsd"].plot(ax=ax, label="Train", style=".", markersize=0.5)
val_data["tsd"].plot(ax=ax, label="Validation", style=".", markersize=0.5)
test_data["tsd"].plot(ax=ax, label="Test", style=".", markersize=0.5)
ax.axvline(THRESHOLD_DATE_1, color="k", ls="--")
ax.axvline(THRESHOLD_DATE_2, color="k", ls=":")
ax.set_title("Train / Validation / Test Split")
ax.set_ylabel("Electricity Demand (MW)")
plt.legend();

In [ ]:
# 定义特征和目标变量
FEATURES = [
    "is_holiday",
    "settlement_period",
    "day_of_month",
    "day_of_week",
    "day_of_year",
    "quarter",
    "month",
    "week_of_year",
    "lag1",
    "lag2",
    "lag3",
]
TARGET = "tsd"

# 需要归一化的列（数值范围大的特征 + 目标）
COLS_TO_SCALE = ["lag1", "lag2", "lag3", TARGET]

X_train = train_data[FEATURES]
y_train = train_data[TARGET]

X_val = val_data[FEATURES]
y_val = val_data[TARGET]

X_test = test_data[FEATURES]
y_test = test_data[TARGET]

print(f"训练集: {X_train.shape}, 验证集: {X_val.shape}, 测试集: {X_test.shape}")

<a id='transformer'></a>
# Transformer 时序预测模型

使用 PyTorch 实现 Transformer 模型。相比 LSTM，Transformer 通过自注意力机制能更好地捕捉长距离依赖关系。

## 数据预处理

需要对数据进行归一化（MinMaxScaler 用 numpy 手动实现），并构建序列窗口用于 Transformer 输入。

In [ ]:
# 部分归一化：只对 lag 和 tsd 做 MinMaxScaling
FEATURES_TARGET = FEATURES + [TARGET]
train_data_np = train_data[FEATURES_TARGET].values.astype(np.float32)
val_data_np = val_data[FEATURES_TARGET].values.astype(np.float32)
test_data_np = test_data[FEATURES_TARGET].values.astype(np.float32)

# 找到需要归一化的列的索引
scale_indices = [FEATURES_TARGET.index(c) for c in COLS_TO_SCALE]

# 在训练集上计算 min 和 max（仅归一化列）
data_min = train_data_np[:, scale_indices].min(axis=0)
data_max = train_data_np[:, scale_indices].max(axis=0)
data_range = data_max - data_min
data_range[data_range == 0] = 1.0

# 只对指定列归一化
train_scaled = train_data_np.copy()
val_scaled = val_data_np.copy()
test_scaled = test_data_np.copy()

train_scaled[:, scale_indices] = (train_data_np[:, scale_indices] - data_min) / data_range
val_scaled[:, scale_indices] = (val_data_np[:, scale_indices] - data_min) / data_range
test_scaled[:, scale_indices] = (test_data_np[:, scale_indices] - data_min) / data_range

# tsd 的 min/range 用于后续反归一化
tsd_scale_idx = COLS_TO_SCALE.index(TARGET)
tsd_min = data_min[tsd_scale_idx]
tsd_range_val = data_range[tsd_scale_idx]

print(f"训练集形状: {train_scaled.shape}")
print(f"验证集形状: {val_scaled.shape}")
print(f"测试集形状: {test_scaled.shape}")
print(f"归一化列: {COLS_TO_SCALE}")

## 构建序列窗口

Transformer 需要序列输入。我们使用滑动窗口将数据组织为 `(batch, seq_len, features)` 的格式。

In [ ]:
def create_sequences(data, seq_len):
    """将数据按滑动窗口切分为序列"""
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len, :-1])   # 特征
        y.append(data[i+seq_len, -1])       # 目标（下一个时间步的 tsd）
    return np.array(X), np.array(y)

X_train_seq, y_train_seq = create_sequences(train_scaled, SEQ_LEN)
X_val_seq, y_val_seq = create_sequences(val_scaled, SEQ_LEN)
X_test_seq, y_test_seq = create_sequences(test_scaled, SEQ_LEN)

print(f"训练序列: X={X_train_seq.shape}, y={y_train_seq.shape}")
print(f"验证序列: X={X_val_seq.shape}, y={y_val_seq.shape}")
print(f"测试序列: X={X_test_seq.shape}, y={y_test_seq.shape}")

In [ ]:
# 转换为 PyTorch 张量并创建 DataLoader
train_dataset = TensorDataset(
    torch.tensor(X_train_seq, dtype=torch.float32),
    torch.tensor(y_train_seq, dtype=torch.float32)
)
val_dataset = TensorDataset(
    torch.tensor(X_val_seq, dtype=torch.float32),
    torch.tensor(y_val_seq, dtype=torch.float32)
)
test_dataset = TensorDataset(
    torch.tensor(X_test_seq, dtype=torch.float32),
    torch.tensor(y_test_seq, dtype=torch.float32)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"训练批次数: {len(train_loader)}, 验证批次数: {len(val_loader)}, 测试批次数: {len(test_loader)}")

## Transformer 模型定义

数据流向：`输入(9维特征) → 线性投影(d_model维) → 位置编码 → Transformer Encoder × N层 → 取最后时间步 → 全连接回归输出`

In [ ]:
class PositionalEncoding(nn.Module):
    """
    正弦-余弦位置编码
    为序列中的每个位置生成唯一的编码向量，让模型能区分不同时间步
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        # 创建位置编码矩阵 (max_len, d_model)
        pe = torch.zeros(max_len, d_model)

        # 位置索引: [0, 1, 2, ..., max_len-1]，形状 (max_len, 1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # 频率缩放因子，控制不同维度的振荡频率
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model)
        )

        # 偶数维度用 sin，奇数维度用 cos
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[:d_model // 2])

        # 增加 batch 维度: (1, max_len, d_model)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        # 将位置编码加到输入上，只取前 seq_len 个位置
        return x + self.pe[:, :x.size(1), :]


class TimeSeriesTransformer(nn.Module):
    """
    基于 Transformer Encoder 的时序预测模型

    数据流:
        输入 (batch, seq_len, n_features)
          ↓ Linear 投影
        (batch, seq_len, d_model)
          ↓ + 位置编码
        (batch, seq_len, d_model)
          ↓ Transformer Encoder × num_layers
        (batch, seq_len, d_model)
          ↓ 取最后一个时间步
        (batch, d_model)
          ↓ 全连接回归头
        (batch,)  → 预测值
    """
    def __init__(self, n_features, d_model, nhead, num_layers, dim_feedforward, dropout):
        super().__init__()

        # 第1层：输入投影，将原始特征维度映射到模型内部维度
        self.input_proj = nn.Linear(n_features, d_model)

        # 第2层：位置编码
        self.pos_enc = PositionalEncoding(d_model)

        # 第3层：Transformer 编码器（堆叠 num_layers 层）
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 第4层：概率输出头（输出均值 μ 和对数方差 log_var）
        self.mu_head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
        self.logvar_head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        # x 形状: (batch, seq_len, n_features)

        # 步骤1: 投影到 d_model 维
        x = self.input_proj(x)    # → (batch, seq_len, d_model)

        # 步骤2: 加上位置编码
        x = self.pos_enc(x)       # → (batch, seq_len, d_model)

        # 步骤3: 通过 Transformer 编码器
        x = self.encoder(x)       # → (batch, seq_len, d_model)

        # 步骤4: 取最后一个时间步的输出作为整个序列的表示
        x = x[:, -1, :]           # → (batch, d_model)

        # 步骤5: 输出均值 μ 和方差 var（概率分布参数）
        mu = self.mu_head(x).squeeze(-1)           # → (batch,)
        log_var = self.logvar_head(x).squeeze(-1)  # → (batch,)
        var = torch.exp(log_var)                    # 保证方差为正

        return mu, var


# 实例化模型
model = TimeSeriesTransformer(
    n_features=len(FEATURES),
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
).to(device)

print(model)
print(f"\n可训练参数量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 模型训练

使用 RMSE 损失函数和 Adam 优化器，并实现早停机制。

In [ ]:
# 高斯负对数似然损失（概率分布输出）
criterion = nn.GaussianNLLLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE
)

# 训练记录
train_losses = []
val_losses = []
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

for epoch in range(NUM_EPOCHS):

    # ---- 训练阶段 ----
    model.train()
    total_train_loss = 0.0
    num_train_batches = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        mu, var = model(X_batch)
        loss = criterion(mu, y_batch, var)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        num_train_batches += 1

    avg_train_loss = total_train_loss / num_train_batches

    # ---- 验证阶段 ----
    model.eval()
    total_val_loss = 0.0
    num_val_batches = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            mu, var = model(X_batch)
            loss = criterion(mu, y_batch, var)

            total_val_loss += loss.item()
            num_val_batches += 1

    avg_val_loss = total_val_loss / num_val_batches

    # ---- 记录损失 ----
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    # ---- 学习率调度 ----
    scheduler.step(avg_val_loss)

    # ---- 早停检查 ----
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    # ---- 打印日志 ----
    if (epoch + 1) % 5 == 0 or patience_counter == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
              f"Train NLL: {avg_train_loss:.6f} | "
              f"Val NLL: {avg_val_loss:.6f} | "
              f"Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at Epoch {epoch+1}")
        break

# 恢复最优模型权重
model.load_state_dict(best_model_state)
model.to(device)
print("Best model weights loaded")

In [ ]:
# 绘制训练和验证损失曲线
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, len(train_losses) + 1), train_losses, label="Train Loss")
ax.plot(range(1, len(val_losses) + 1), val_losses, label="Validation Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Gaussian NLL Loss")
ax.set_title("Training and Validation Loss")
ax.legend(loc="best");

## 预测与反归一化

模型训练在归一化数据上进行，因此需要将预测结果反归一化回原始尺度。

In [ ]:
# 在测试集上预测
model.eval()
mu_list = []
std_list = []

with torch.no_grad():
    for X_batch, _ in test_loader:
        X_batch = X_batch.to(device)
        mu, var = model(X_batch)
        mu_list.append(mu.cpu().numpy())
        std_list.append(torch.sqrt(var).cpu().numpy())

# 拼接所有 batch 的预测结果
pred_scaled = np.concatenate(mu_list)
std_scaled = np.concatenate(std_list)

# 反归一化：将 [0,1] 范围的预测值还原到原始尺度
pred_original = pred_scaled * tsd_range_val + tsd_min
std_original = std_scaled * tsd_range_val

# 构建结果 DataFrame
result_frame = y_test.iloc[SEQ_LEN:].to_frame()
result_frame["pred_transformer"] = pred_original
result_frame["std_transformer"] = std_original

print(f"预测结果: {len(pred_original)} 个样本")

<a id='results'></a>
# 结果分析

## 测试集预测可视化

In [ ]:
# 测试集全局预测对比（含 95% 置信区间）
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(result_frame.index, result_frame["tsd"], "o", markersize=1, alpha=0.3, label="Actual")
ax.plot(result_frame.index, result_frame["pred_transformer"], "o", markersize=1, alpha=0.3, label="Prediction (μ)")
ax.fill_between(
    result_frame.index,
    result_frame["pred_transformer"] - 1.96 * result_frame["std_transformer"],
    result_frame["pred_transformer"] + 1.96 * result_frame["std_transformer"],
    alpha=0.15, color="orange", label="95% Confidence Interval"
)
ax.legend(loc="center", bbox_to_anchor=(1.15, 0.5))
ax.set_title("Test Set Predictions with Uncertainty")
ax.set_ylabel("Electricity Demand (MW)")
ax.set_xlabel("Date");

### 两周细节对比

选取一段两周的时间窗口，观察模型对日内模式的捕捉能力。

In [ ]:
begin = "08-01-2024"
end = "08-14-2024"

mask = (result_frame.index > begin) & (result_frame.index < end)

fig, ax = plt.subplots(figsize=(20, 5))
ax.plot(result_frame.loc[mask].index, result_frame.loc[mask]["tsd"], "-o", label="Actual")
ax.plot(result_frame.loc[mask].index, result_frame.loc[mask]["pred_transformer"], "-s", label="Prediction (μ)")
ax.fill_between(
    result_frame.loc[mask].index,
    result_frame.loc[mask]["pred_transformer"] - 1.96 * result_frame.loc[mask]["std_transformer"],
    result_frame.loc[mask]["pred_transformer"] + 1.96 * result_frame.loc[mask]["std_transformer"],
    alpha=0.25, color="orange", label="95% CI"
)
ax.legend(loc="center", bbox_to_anchor=(1.15, 0.5))
ax.set_title("Test Set Predictions — Two-Week Detail")
ax.set_ylabel("Electricity Demand (MW)")
ax.set_xlabel("Date");

通过折线图可以确认 Transformer 预测不是原始数据的简单滞后版本，而是真正捕捉到了日内和周内的用电模式。

## 性能指标

In [ ]:
mape_transformer = mean_absolute_percentage_error(
    result_frame["tsd"], result_frame["pred_transformer"]
)

rmse_transformer = np.sqrt(np.mean(
    (result_frame["tsd"].values - result_frame["pred_transformer"].values) ** 2
))

print(f"Transformer 模型 MAPE: {mape_transformer:.2f}%")
print(f"Transformer 模型 RMSE: {rmse_transformer:.2f} MW")

## 结果总结

In [ ]:
summary_df = pd.DataFrame({
    "Transformer": [mape_transformer, rmse_transformer],
}, index=["MAPE (%)", "RMSE (MW)"])

summary_df.style.format('{:.2f}')

## 总结与展望

**Transformer** 模型利用自注意力机制，能够有效捕捉电力需求数据中的多种季节性模式（日内、周内、年内）。

后续改进方向：
* 尝试不同的序列长度（如 96 或 336 个时间步）
* 增加 Transformer 层数或注意力头数
* 使用概率预测输出（如高斯似然）量化预测的不确定性
* 探索不同的训练-测试划分策略

In [ ]:
print(f"Train losses: {train_losses[:10]}")
print(f"Val losses: {val_losses[:10]}")
print(f"Total epochs: {len(train_losses)}")
print(f"Best val loss: {best_val_loss:.6f}")
print(f"Final LR: {optimizer.param_groups[0]['lr']}")
